# Saltwater intrusion in the advanced coastal valley

The advanced synthetic valley routes its stream with streamflow routing
(**SFR**), holds its lake in a lake (**LAK**) package, moves recharge through
the unsaturated zone (**UZF**), pumps from multi-aquifer wells (**MAW**), and
routes water between them with the water mover (**MVR**). Put a sea along its
southern edge and the same saltwater wedge forms as in the base valley, but now
the stream and the wells are the freshwater features the salt can reach.

## What this notebook covers

Couple flow and transport on the advanced coastal valley, let the Buoyancy
(**BUY**) package make the seawater heavier than the fresh water, and follow
where the salt goes.

By the end of this notebook you will be able to:

- couple a transport model to a flow model that uses the advanced packages,
- say why that needs a water mover transport (**MVT**) package but not SFT,
  LKT, or UZT,
- read the salt that leaves through the stream and the wells from the transport
  budget, and
- compare the wedge under the advanced valley with the one under the base
  valley.

The base valley is covered in
[`mf6-coastal-density`](mf6-coastal-density.ipynb), and the coastal boundary
itself in [`mf6-coastal-ghb`](mf6-coastal-ghb.ipynb).

Import the packages this notebook uses. `mf6_coastal_density` holds the model
builder and the readers, so the cells below stay about the hydrology.

In [ ]:
%matplotlib inline

import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import mf6_coastal_density as cd
import numpy as np
import pandas as pd
from flopy.utils.postprocessing import get_specific_discharge
from mf6_notebook_helpers import find_mf6_libraries

_, mf6_exe = find_mf6_libraries()

## How density enters the flow equation

MODFLOW 6 solves for hydraulic head, and the BUY package turns a simulated
concentration into a fluid density with a straight line:

```
density = denseref + drhodc * concentration
```

`denseref` is the density of fresh water and `drhodc` is the slope. The values
below put seawater at 1,024.5 kg/m3 against 1,000 for fresh water. Only the
ratio of the two enters the flow equation, so the concentration unit does not
have to match the feet and days the valley is built in - but it does mean the
transport budget is in those concentration units times cubic feet, which is not
a physical mass.

In [ ]:
print(f"fresh water:  {cd.DENSEREF:8.1f} kg/m3")
print(f"seawater:     {cd.DENSEREF + cd.DRHODC * cd.SEAWATER:8.1f} kg/m3")
print(f"density ratio: {(cd.DENSEREF + cd.DRHODC * cd.SEAWATER) / cd.DENSEREF:7.4f}")
print(f"porosity:     {cd.POROSITY:8.2f}")
print(f"dispersivity: {cd.ALH:8.1f} ft along flow, {cd.ATH1:.1f} ft across it")

## Coupling transport to the advanced packages

Every stress package a flow model carries has to tell the transport model what
concentration its water arrives at. For ordinary packages the Source and Sink
Mixing (**SSM**) package does that job, and it does the same for SFR, LAK, UZF,
and MAW as long as no advanced transport package has claimed them. Water
entering the aquifer from the stream, the lake, or the unsaturated zone arrives
fresh, and water leaving takes the salinity of the cell it leaves from.

The mover is the one package that has to be named. MODFLOW 6 stops with

```
GWF water mover is active but the GWT MVT package has not been specified.
```

unless a **MVT** package exists, so the builder adds one with no options. The
mover then carries water between the advanced packages without carrying solute,
which is exact here because everything it routes is fresh.

What this leaves out is in-stream and in-lake transport: the stream and the lake
do not hold a salinity of their own, so salt that reaches them leaves the model
rather than travelling down the channel. Simulating that needs the streamflow
transport (**SFT**), lake transport (**LKT**), and unsaturated-zone transport
(**UZT**) packages, one per advanced package, and their own concentration data.

## Build and run the coupled model

`build_simulation()` loads the shipped coastal advanced model, shortens it to a
steady spin-up plus five pumping years, and adds the transport model, the
GWF-GWT exchange, and BUY. The spin-up lets the wedge reach the position the
freshwater discharge holds it at, and the pumping years then move it.

In [ ]:
ws = pl.Path("models/coastal-density-advanced")
sim = cd.build_simulation(ws, mf6_exe, variant="advanced")
success, buff = sim.run_simulation(silent=True)
if not success:
    raise RuntimeError("\n".join(buff[-20:]))

gwf = sim.get_model("sv")
print(f"stress periods: {sim.tdis.nper.data}")
print(f"spin-up steps:  {cd.SPINUP_NSTP}, steps per year after it: {cd.ANNUAL_NSTP}")

## The wedge

Cut a section down the middle of the valley, along a column, and plot salinity
at the end of the spin-up and after each pumping year. The coast is on the left
of each panel, at row 40.

In [ ]:
section_column = 19  # zero-based
conc = cd.concentration(ws)
times = [0, 1, 3, 5]  # spin-up, then years 1, 3, and 5

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"], ["C", "D"]], figsize=(9.5, 6.5), layout="constrained"
    )
    for letter, itime in zip("ABCD", times):
        ax = axd[letter]
        xs = flopy.plot.PlotCrossSection(
            model=gwf, ax=ax, line={"column": section_column}
        )
        cb = xs.plot_array(conc[itime], cmap="viridis", vmin=0.0, vmax=cd.SEAWATER)
        xs.plot_grid(lw=0.2, color="0.8")
        cs = xs.contour_array(
            conc[itime], levels=[0.5 * cd.SEAWATER], colors="w", linewidths=1.5
        )
        label = "end of spin-up" if itime == 0 else f"year {itime}"
        ax.set_title(f"{letter}. {label}")
        ax.set_xlabel("distance from the north end, in feet")
        ax.set_ylabel("elevation, in feet")
        ax.set_xlim(12000.0, 20000.0)
    fig.colorbar(cb, ax=list(axd.values()), shrink=0.6, label="salinity, in kg/m3")

**What to look for.** Salt water fills the deep cells at the coast and thins
inland, which is the wedge. The white line is the 50 percent isochlor, the
contour halfway between fresh water and seawater, and it is the line usually
used to say how far the wedge reaches. The wedge sits under the fresh water
rather than mixing with it, because it is denser.

## Where the sea water goes in and comes out

A coastal boundary is not simply a drain. Sum the boundary flow by layer: the
deep cells take sea water in and the shallow cells send fresh water out, which
is the circulation cell a density model produces and a constant-density model
cannot.

In [ ]:
budget = flopy.utils.CellBudgetFile(ws / "sv.cbc")
ghb = budget.get_data(text="GHB", totim=budget.get_times()[-1])[0]
ncell = gwf.dis.nrow.data * gwf.dis.ncol.data
by_layer = np.zeros(gwf.dis.nlay.data)
for node, q in zip(ghb["node"], ghb["q"]):
    by_layer[(node - 1) // ncell] += q

for k, q in enumerate(by_layer):
    direction = "into the aquifer" if q > 0 else "out to the sea"
    print(f"layer {k + 1}: {q:12,.0f} ft3/d {direction}")
print(f"total:   {by_layer.sum():12,.0f} ft3/d")

Follow the wedge and the salt through time. The toe is the farthest inland cell
whose salinity is at least half that of seawater, measured along the section
column.

In [ ]:
delc = float(np.atleast_1d(gwf.dis.delc.array)[0])
toe = [
    cd.wedge_toe(np.nan_to_num(conc[i]), section_column, gwf.dis.nlay.data - 1, delc)
    for i in range(conc.shape[0])
]
salt = pd.read_csv(ws / f"{cd.GWT_NAME}-budget.csv")

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(9.5, 3.6), layout="constrained"
    )

    ax = axd["A"]
    ax.plot(range(len(toe)), toe, "o-", color="tab:blue")
    ax.set_xlabel("stress period")
    ax.set_ylabel("toe position, in feet inland")
    ax.set_title("A. How far the wedge reaches")

    ax = axd["B"]
    ax.plot(salt["time"] / 365.25, salt["STORAGE-AQUEOUS(MST)_IN"], color="tab:red")
    ax.set_xlabel("time, in years")
    ax.set_ylabel("salt into storage, in kg/m3 x ft3/d")
    ax.set_title("B. Salt accumulating in the aquifer")

## What density is responsible for

Run the same model again with the BUY package left out. Everything else is
identical, so the difference between the two runs is what the density contrast
does.

In [ ]:
ws_fresh = pl.Path(f"{ws}-no-buoyancy")
sim_fresh = cd.build_simulation(ws_fresh, mf6_exe, variant="advanced", buoyancy=False)
success, buff = sim_fresh.run_simulation(silent=True)
if not success:
    raise RuntimeError("\n".join(buff[-20:]))

head = flopy.utils.HeadFile(ws / "sv.hds").get_alldata()[-1]
head_fresh = flopy.utils.HeadFile(ws_fresh / "sv.hds").get_alldata()[-1]
conc_fresh = cd.concentration(ws_fresh)

print(
    f"deepest-layer head, coastal row, with density:    {head[-1, -1].mean():8.3f} ft"
)
print(
    f"deepest-layer head, coastal row, without density: {head_fresh[-1, -1].mean():8.3f} ft"
)
print(
    f"largest head difference anywhere:                 {np.nanmax(np.abs(head - head_fresh)):8.3f} ft"
)
print(
    f"toe with density:    {cd.wedge_toe(np.nan_to_num(conc[-1]), section_column, gwf.dis.nlay.data - 1, delc):8.0f} ft"
)
print(
    f"toe without density: {cd.wedge_toe(np.nan_to_num(conc_fresh[-1]), section_column, gwf.dis.nlay.data - 1, delc):8.0f} ft"
)

**What to look for.** Without buoyancy the salt still enters, because the
boundary still carries it, but it spreads as a plume rather than settling into a
wedge, and the heads along the coast differ. The density term is what keeps the
salt water at the bottom of the aquifer and drives the circulation.

## Which way the water moves at the coast

A coast is usually drawn as a discharge zone, with fresh water rising through the
aquifer and leaving at the shore. Whether this model does that depends on
density. Plot the vertical specific discharge along the valley for both runs,
positive upward, in a shallow layer and a deep one.

In [ ]:
def vertical_flow(workspace, layer):
    """Mean vertical specific discharge by row in one layer, positive upward."""
    cbc = flopy.utils.CellBudgetFile(workspace / "sv.cbc")
    totim = cbc.get_times()[-1]
    head = flopy.utils.HeadFile(workspace / "sv.hds").get_data(totim=totim)
    spdis = cbc.get_data(text="DATA-SPDIS", totim=totim)[0]
    _, _, qz = get_specific_discharge(spdis, gwf, head=head)
    return np.nanmean(qz[layer], axis=1)


# what a constant-density model does when the weight of the sea is put in by
# hand, as a boundary head that rises with depth
ws_efh = pl.Path(f"{ws}-freshwater-head")
sim_efh = cd.build_simulation(
    ws_efh, mf6_exe, variant="advanced", buoyancy=False, equivalent_freshwater=True
)
success, buff = sim_efh.run_simulation(silent=True)
if not success:
    raise RuntimeError("\n".join(buff[-20:]))

runs = (
    ("with density", ws, "tab:blue"),
    ("constant density, sea level", ws_fresh, "tab:gray"),
    ("constant density, freshwater head", ws_efh, "tab:red"),
)
inland = (gwf.dis.nrow.data - 1 - np.arange(gwf.dis.nrow.data)) * delc

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]], figsize=(9.5, 3.8), layout="constrained"
    )
    for letter, layer in zip("AB", (1, 3)):
        ax = axd[letter]
        for label, workspace, color in runs:
            ax.plot(inland, vertical_flow(workspace, layer), color=color, label=label)
        ax.axhline(0.0, lw=0.8, color="k")
        ax.set_xlim(0.0, 10000.0)
        # the freshwater-head run leaves the top of both panels; scaling to it
        # would flatten the other two, so annotate its peak instead
        ax.set_ylim(-0.012, 0.012)
        peak = vertical_flow(ws_efh, layer)[-1]
        ax.annotate(
            f"{peak:.3f} ft/d at the coast",
            xy=(0.0, 0.012),
            xytext=(1200.0, 0.0085),
            color="tab:red",
            fontsize=7,
            arrowprops={"arrowstyle": "->", "color": "tab:red", "lw": 0.8},
        )
        ax.set_xlabel("distance inland from the coast, in feet")
        ax.set_ylabel("vertical specific discharge, in feet per day")
        ax.set_title(f"{letter}. Layer {layer + 1}")
    axd["A"].legend(fontsize=7, loc="lower right")

**What to look for.** The three runs disagree about which way the water moves at
the shore, which is the whole argument for simulating density.

With a sea-level boundary and no density the water moves down all the way from
about 5,500 ft inland to the coast, at -0.0023 ft/d in layer 4 at the shore.
Most of the coastal conductance is in the two thickest layers, so the deep
aquifer is the easiest way out and nothing rises at all. The upward band farther
inland, 6,000 to 8,500 ft out, belongs to the river rather than to the sea.

With density the near-shore flow reverses and layer 4 rises at 0.0088 ft/d at the
coast. That is the circulation cell in profile: sea water sinking in at depth,
mixing, and leaving as brackish water through the shallow layers.

Carrying the weight of the sea by hand, as a boundary head that rises with depth,
turns the flow upward as well - but at 0.176 ft/d, twenty times the rate the
density model gives. The equivalent freshwater head assumes salt water fills the
aquifer to its base, so it overstates the pressure at depth. It gets the
direction right and the magnitude wrong.

The freshwater-head run runs off the top of both panels; its value at the coast
is annotated on each.


## Where the salt leaves

The transport budget has one entry per flow package, so it says which features
the salt leaves through. The stream and the wells are the ones that matter here.

In [ ]:
salt_last = salt.iloc[-1]
for column in salt.columns:
    if "SSM" in column and column.endswith("_OUT") and salt_last[column] > 0.0:
        print(f"{column:28s} {salt_last[column]:15,.0f}")

**What to look for.** Salt leaves through the stream and through the wells,
which is the practical result of a coastal aquifer: pumping near a coast draws
salt toward the wells, and a stream in hydraulic contact with a salty aquifer
carries that salt away. The numbers are in concentration units times volume, so
read them against each other rather than as kilograms.

## Recap

- The BUY package turns simulated salinity into fluid density with a reference
  density and a slope, and the flow and transport models are solved together
  through a GWF-GWT exchange.
- Sea water enters the aquifer through the deep coastal cells and fresh water
  leaves through the shallow ones, which is the circulation cell that a
  constant-density model cannot produce.
- The 50 percent isochlor is the usual measure of how far the wedge reaches.
- Running the same model without buoyancy separates what the density contrast
  does from what the boundary does.
- SSM handles SFR, LAK, UZF, and MAW as ordinary boundaries, so no SFT, LKT, or
  UZT package is needed; the mover needs an MVT package, which can be empty when
  everything the mover routes is fresh.